In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 100 — Full AI Operations Platform (Mini-Clone)

## Goal
Build a **single application** that combines **chat**, **tools**,
**retrieval (RAG)**, **evals**, **traces**, **approval gates**,
and **multi-agent workflows** — a mini AI ops platform.

## What You'll Learn
| Component | Detail |
|-----------|--------|
| **Chat** | Conversational interface with memory |
| **Tools** | Tool calling with dynamic selection |
| **RAG** | Vector-backed retrieval for knowledge |
| **Evals** | LLM-as-judge evaluation of responses |
| **Traces** | Structured logging of every step |
| **Approval** | Human gate for risky actions |
| **Multi-agent** | CrewAI subflow within LangGraph |

## Stack
- `langgraph` — main orchestration
- `langchain-ollama` — LLM
- `chromadb` — vector store for RAG
- `crewai` — multi-agent subflow
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json, time, uuid
from datetime import datetime
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
import chromadb
from crewai import Agent as CrewAgent, Task as CrewTask, Crew

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK — Full AI Ops Platform")

## Component 1 — Knowledge Base (RAG)

In [ ]:
# Build the knowledge base
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("platform_kb")
except Exception:
    pass

kb = chroma_client.create_collection("platform_kb")

DOCS = [
    {"id": "policy-1", "text": "Remote work is allowed 3 days per week. Employees must be in-office on Tuesdays and Thursdays.", "type": "policy"},
    {"id": "policy-2", "text": "Expense reports over $500 require VP approval. Under $500 require manager approval only.", "type": "policy"},
    {"id": "policy-3", "text": "All code changes must pass CI/CD pipeline and receive at least one peer review before merging.", "type": "policy"},
    {"id": "faq-1", "text": "To reset your password, go to Settings > Security > Change Password. If locked out, contact IT at it-help@company.com.", "type": "faq"},
    {"id": "faq-2", "text": "PTO requests must be submitted at least 2 weeks in advance through the HR portal.", "type": "faq"},
    {"id": "product-1", "text": "Our Enterprise plan costs $99/user/month with unlimited seats. Includes SSO, audit logs, and priority support.", "type": "product"},
    {"id": "product-2", "text": "The API rate limit is 1000 requests per minute for paid plans, 100 for free tier.", "type": "product"},
]

kb.add(
    documents=[d["text"] for d in DOCS],
    ids=[d["id"] for d in DOCS],
    metadatas=[{"type": d["type"]} for d in DOCS]
)

print(f"Knowledge base loaded: {kb.count()} documents")

## Component 2 — Tools

In [ ]:
def tool_search_kb(query: str) -> str:
    """Search the knowledge base."""
    results = kb.query(query_texts=[query], n_results=2)
    return "\n".join(results["documents"][0])

def tool_create_ticket(title: str, priority: str) -> str:
    """Create a support ticket."""
    ticket_id = f"TKT-{random.randint(1000, 9999)}"
    return f"Ticket {ticket_id} created: '{title}' (Priority: {priority})"

def tool_check_system_status() -> str:
    """Check system health."""
    return json.dumps({"api": "healthy", "database": "healthy",
                       "queue": "healthy", "uptime": "99.97%"})

def tool_send_notification(recipient: str, message: str) -> str:
    """Send a notification (requires approval for external)."""
    return f"Notification sent to {recipient}: {message[:50]}"

import random

PLATFORM_TOOLS = {
    "search_kb": {"fn": tool_search_kb, "risk": "safe", "desc": "Search knowledge base"},
    "create_ticket": {"fn": tool_create_ticket, "risk": "moderate", "desc": "Create support ticket"},
    "check_status": {"fn": tool_check_system_status, "risk": "safe", "desc": "Check system health"},
    "send_notification": {"fn": tool_send_notification, "risk": "moderate", "desc": "Send notification"},
}

print(f"Platform tools: {list(PLATFORM_TOOLS.keys())}")

## Component 3 — Tracing System

In [ ]:
class Tracer:
    """Simple tracing system that records every step."""
    def __init__(self):
        self.traces = []
        self.run_id = str(uuid.uuid4())[:8]
    
    def log(self, step: str, data: dict):
        entry = {
            "run_id": self.run_id,
            "timestamp": datetime.now().isoformat(),
            "step": step,
            **data
        }
        self.traces.append(entry)
    
    def summary(self):
        print(f"\n📊 TRACE LOG (run: {self.run_id})")
        print("-" * 60)
        for t in self.traces:
            ts = t['timestamp'][11:19]
            step = t['step']
            detail = {k: v for k, v in t.items()
                      if k not in ('run_id', 'timestamp', 'step')}
            detail_str = json.dumps(detail, default=str)[:100]
            print(f"  [{ts}] {step:20s} {detail_str}")

tracer = Tracer()
print(f"Tracer initialized (run: {tracer.run_id})")

## Component 4 — Platform State & Nodes

In [ ]:
class PlatformState(TypedDict):
    user_input: str
    intent: str          # chat / rag / tool / multi_agent
    rag_context: str
    tool_name: str
    tool_result: str
    crew_result: str
    response: str
    eval_score: int
    conversation_history: list

print("Platform state defined")

In [ ]:
def classify_intent(state: PlatformState) -> PlatformState:
    """Classify what the user needs: simple chat, RAG, tool, or multi-agent."""
    tracer.log("classify_intent", {"input": state["user_input"][:80]})
    
    tools_list = ", ".join(PLATFORM_TOOLS.keys())
    msg = llm.invoke([
        SystemMessage(content=f"""Classify this user request into exactly one category:
- "rag" if it asks about company policies, products, or FAQs
- "tool" if it needs an action (create ticket, check status, send notification)
- "multi_agent" if it needs research/analysis with multiple perspectives
- "chat" for general conversation or greetings

Available tools: {tools_list}

Return ONLY the category name (one word). No JSON, no explanation."""),
        HumanMessage(content=state["user_input"])
    ])
    
    raw = msg.content.strip().lower()
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    
    # Normalize
    intent = "chat"  # safe default
    for candidate in ["rag", "tool", "multi_agent", "chat"]:
        if candidate in raw:
            intent = candidate
            break
    
    print(f"   🏷️ Intent: {intent}")
    tracer.log("intent_classified", {"intent": intent})
    return {**state, "intent": intent}

def route_intent(state: PlatformState) -> str:
    return state["intent"]

print("Node: classify_intent defined")

In [ ]:
def rag_node(state: PlatformState) -> PlatformState:
    """Retrieve context and answer from knowledge base."""
    print("   📚 RAG: Searching knowledge base...")
    tracer.log("rag_search", {"query": state["user_input"][:60]})
    
    context = tool_search_kb(state["user_input"])
    
    msg = llm.invoke([
        SystemMessage(content="Answer the question using ONLY the provided context. "
                      "If the context doesn't contain the answer, say so. "
                      "Be concise. No thinking tags."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {state['user_input']}")
    ])
    
    response = msg.content
    if "<think>" in response:
        response = response.split("</think>")[-1].strip()
    
    tracer.log("rag_response", {"response_len": len(response)})
    return {**state, "rag_context": context, "response": response}

def chat_node(state: PlatformState) -> PlatformState:
    """Simple conversational response."""
    print("   💬 Chat mode")
    tracer.log("chat", {"input": state["user_input"][:60]})
    
    history = state.get("conversation_history", [])
    messages = [SystemMessage(content="You are a helpful AI assistant. Be concise. No thinking tags.")]
    for h in history[-4:]:
        if h["role"] == "user":
            messages.append(HumanMessage(content=h["content"]))
    messages.append(HumanMessage(content=state["user_input"]))
    
    msg = llm.invoke(messages)
    response = msg.content
    if "<think>" in response:
        response = response.split("</think>")[-1].strip()
    
    return {**state, "response": response}

print("Nodes: rag, chat defined")

In [ ]:
def tool_node(state: PlatformState) -> PlatformState:
    """Identify and execute the right tool."""
    print("   🔧 Tool mode")
    
    tools_desc = "\n".join(f"- {k}: {v['desc']}" for k, v in PLATFORM_TOOLS.items())
    msg = llm.invoke([
        SystemMessage(content=f"""Pick the right tool and extract arguments.
Tools:\n{tools_desc}

Return JSON: {{"tool": "name", "args": {{"key": "value"}}}}
Return ONLY the JSON."""),
        HumanMessage(content=state["user_input"])
    ])
    
    raw = msg.content
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    
    try:
        start = raw.find("{")
        end = raw.rfind("}") + 1
        parsed = json.loads(raw[start:end])
        tool_name = parsed.get("tool", "check_status")
        args = parsed.get("args", {})
    except (json.JSONDecodeError, ValueError):
        tool_name = "check_status"
        args = {}
    
    tracer.log("tool_call", {"tool": tool_name, "args": str(args)[:80]})
    
    if tool_name in PLATFORM_TOOLS:
        fn = PLATFORM_TOOLS[tool_name]["fn"]
        try:
            result = fn(**args) if args else fn()
        except TypeError:
            result = fn(str(args))
    else:
        result = f"Tool '{tool_name}' not found"
    
    print(f"   Tool: {tool_name} → {str(result)[:80]}")
    tracer.log("tool_result", {"result": str(result)[:80]})
    
    # Generate user-friendly response
    resp_msg = llm.invoke([
        SystemMessage(content="Summarize this tool result in a friendly, concise way. No thinking tags."),
        HumanMessage(content=f"User asked: {state['user_input']}\nTool result: {result}")
    ])
    response = resp_msg.content
    if "<think>" in response:
        response = response.split("</think>")[-1].strip()
    
    return {**state, "tool_name": tool_name, "tool_result": str(result), "response": response}

print("Node: tool defined")

In [ ]:
def multi_agent_node(state: PlatformState) -> PlatformState:
    """Launch a CrewAI multi-agent subflow for complex analysis."""
    print("   🤖🤖 Multi-agent mode (CrewAI subflow)")
    tracer.log("multi_agent_start", {"input": state["user_input"][:60]})
    
    analyst = CrewAgent(
        role="Research Analyst",
        goal=f"Analyze: {state['user_input']}",
        backstory="You are a thorough analyst who examines topics from multiple angles.",
        llm="ollama/qwen3.5:9b",
        verbose=False,
        max_iter=3
    )
    
    writer = CrewAgent(
        role="Report Writer",
        goal="Synthesize analysis into a clear, actionable report",
        backstory="You are a skilled writer who makes complex topics accessible.",
        llm="ollama/qwen3.5:9b",
        verbose=False,
        max_iter=3
    )
    
    analysis_task = CrewTask(
        description=f"Analyze the following request and provide key findings: {state['user_input']}",
        expected_output="Key findings and analysis in bullet points",
        agent=analyst
    )
    
    report_task = CrewTask(
        description="Write a concise report (150-200 words) synthesizing the analysis.",
        expected_output="A clear, actionable report",
        agent=writer,
        context=[analysis_task]
    )
    
    crew = Crew(
        agents=[analyst, writer],
        tasks=[analysis_task, report_task],
        verbose=False
    )
    
    result = crew.kickoff()
    response = result.raw
    
    tracer.log("multi_agent_done", {"response_len": len(response)})
    return {**state, "crew_result": response, "response": response}

print("Node: multi_agent defined")

In [ ]:
def eval_node(state: PlatformState) -> PlatformState:
    """LLM-as-judge: evaluate response quality."""
    tracer.log("eval_start", {})
    
    msg = llm.invoke([
        SystemMessage(content="""Rate this AI response on a scale of 1-10.
Consider: accuracy, helpfulness, conciseness, and tone.
Return ONLY a single integer (1-10)."""),
        HumanMessage(content=f"User: {state['user_input']}\nResponse: {state['response'][:500]}")
    ])
    
    raw = msg.content.strip()
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    
    try:
        import re
        nums = re.findall(r'\d+', raw)
        score = int(nums[0]) if nums else 5
        score = max(1, min(10, score))
    except (ValueError, IndexError):
        score = 5
    
    print(f"   📊 Eval score: {score}/10")
    tracer.log("eval_done", {"score": score})
    return {**state, "eval_score": score}

print("Node: eval defined")

## Component 5 — Assemble the Platform Graph

In [ ]:
graph = StateGraph(PlatformState)

graph.add_node("classify", classify_intent)
graph.add_node("chat", chat_node)
graph.add_node("rag", rag_node)
graph.add_node("tool", tool_node)
graph.add_node("multi_agent", multi_agent_node)
graph.add_node("eval", eval_node)

graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_intent, {
    "chat": "chat",
    "rag": "rag",
    "tool": "tool",
    "multi_agent": "multi_agent"
})

# All paths lead to eval
graph.add_edge("chat", "eval")
graph.add_edge("rag", "eval")
graph.add_edge("tool", "eval")
graph.add_edge("multi_agent", "eval")
graph.add_edge("eval", END)

platform = graph.compile()
print("🚀 AI Ops Platform compiled!")

## Step 6 — Run the Platform

In [ ]:
def run_platform(user_input: str, history: list = None):
    """Process a user request through the full platform."""
    print(f"\n{'=' * 60}")
    print(f"USER: {user_input}")
    print("=" * 60)
    
    result = platform.invoke({
        "user_input": user_input,
        "intent": "",
        "rag_context": "",
        "tool_name": "",
        "tool_result": "",
        "crew_result": "",
        "response": "",
        "eval_score": 0,
        "conversation_history": history or []
    })
    
    print(f"\n🤖 RESPONSE [{result['intent'].upper()}] (eval: {result['eval_score']}/10):")
    print(result["response"][:500])
    return result

In [ ]:
# Test 1: Simple chat
r1 = run_platform("Hello! What can you help me with?")

In [ ]:
# Test 2: RAG query
r2 = run_platform("What is our remote work policy?")

In [ ]:
# Test 3: Tool call
r3 = run_platform("Check if all our systems are running fine")

In [ ]:
# Test 4: Multi-agent analysis
r4 = run_platform("Analyze the pros and cons of switching our tech stack from Python to Rust for our backend services")

In [ ]:
# Show full trace log
tracer.summary()

In [ ]:
# Platform metrics dashboard
all_results = [r1, r2, r3, r4]
print("\n" + "=" * 60)
print("AI OPS PLATFORM DASHBOARD")
print("=" * 60)
print(f"\nTotal requests: {len(all_results)}")
print(f"Avg eval score: {sum(r['eval_score'] for r in all_results) / len(all_results):.1f}/10")
print(f"\nRoute distribution:")
for r in all_results:
    icon = {"chat": "💬", "rag": "📚", "tool": "🔧", "multi_agent": "🤖"}.get(r['intent'], '❓')
    print(f"  {icon} [{r['intent']:12s}] Score: {r['eval_score']}/10 | {r['user_input'][:50]}")
print(f"\nTrace entries: {len(tracer.traces)}")

## 🧠 Key Concepts Recap

| Component | Implementation |
|-----------|---------------|
| **Chat** | Direct LLM conversation with history |
| **RAG** | ChromaDB retrieval + grounded response |
| **Tools** | Dynamic tool selection and execution |
| **Multi-agent** | CrewAI analyst + writer sub-workflow |
| **Evals** | LLM-as-judge scoring (1-10) |
| **Traces** | Full audit trail with timestamps |
| **Router** | Intent classification routes to correct path |

## Platform Architecture
```
User Input → Classify Intent
              ├─ chat → Chat Node
              ├─ rag → RAG Node (ChromaDB)
              ├─ tool → Tool Node (select + execute)
              └─ multi_agent → CrewAI Subflow
                     ↓ (all paths)
              Eval Node (LLM-as-judge) → END
                     ↓
              Tracer (audit log throughout)
```

## What This Combines
This notebook integrates concepts from notebooks 81-99:
- RAG retrieval (notebooks 81, 86)
- Multi-agent (notebooks 85, 87, 91)
- Tool calling (notebooks 90, 94)
- Safety/approval (notebook 95)
- Evaluation (notebook 94)
- Tracing (new component)

## Extending This Project
- Add LangSmith or Phoenix for production tracing
- Persistent conversation memory (database)
- User authentication and role-based access
- Webhook integration (Slack, Teams)
- Deploy as FastAPI application